# k-Means — clustering as coordinate descent

> Tutorial pair for [`kmeans.py`](kmeans.py).

## 1. Intuition
Find $k$ "prototypes" (centroids) so that every point is close to its nearest
prototype. Repeat two cheap steps until nothing moves: **assign** points to the
nearest centroid, then **recompute** each centroid as the mean of its points.

## 2. Concept (the slide)
- **Objective (inertia):** minimize within-cluster squared distance
  $J=\sum_{i}\lVert \mathbf x_i-\boldsymbol\mu_{c_i}\rVert^2$.
- **Lloyd's algorithm:** alternate assignment (E) and centroid update (M).
- **Caveats:** finds a *local* optimum (init-sensitive), assumes roughly
  spherical, equally-sized clusters, and you must choose $k$.

## 3. Math derivation

Minimize over both assignments $c_i\in\{1..k\}$ and centroids $\boldsymbol\mu_j$:

$$J(\{c_i\},\{\boldsymbol\mu_j\})=\sum_{i=1}^{n}\lVert \mathbf x_i-\boldsymbol\mu_{c_i}\rVert^2 .$$

This is **coordinate descent**, decreasing $J$ on each half-step:

- **Assignment (fix $\mu$, optimize $c$):** each point independently picks
  $c_i=\arg\min_j\lVert\mathbf x_i-\boldsymbol\mu_j\rVert^2$ — clearly minimizes its term.
- **Update (fix $c$, optimize $\mu$):** set $\partial J/\partial\boldsymbol\mu_j=0$:
  $$\sum_{i:c_i=j}2(\boldsymbol\mu_j-\mathbf x_i)=0
   \;\Rightarrow\;
   \boxed{\;\boldsymbol\mu_j=\frac{1}{|C_j|}\sum_{i\in C_j}\mathbf x_i\;}$$
  the centroid is the **mean** (hence the name).

Both steps never increase $J$, and there are finitely many assignments, so the
algorithm **converges** — but only to a local minimum.

**k-means++ initialization.** Pick the first centroid at random, then pick each
next centroid with probability $\propto D(\mathbf x)^2$ (squared distance to the
nearest chosen centroid). This spreads seeds out and gives an
$O(\log k)$-competitive expected cost — far better than random seeds.

**Choosing $k$.** *Elbow*: plot $J$ vs $k$, look for the kink. *Silhouette*:
$s_i=\dfrac{b_i-a_i}{\max(a_i,b_i)}$ where $a_i$ = mean intra-cluster distance,
$b_i$ = mean distance to the nearest *other* cluster; average $s_i$ near 1 is good.

## 4. NumPy implementation (Lloyd + k-means++ + mini-batch + silhouette)

In [ ]:
# ===== actual implementation from kmeans.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def _dist2(X, C):
    """Squared Euclidean distance, X:(n,d) C:(k,d) -> (n,k)."""
    return ((X[:, None, :] - C[None, :, :]) ** 2).sum(2)

def silhouette_score(X, labels):
    """Mean silhouette: (b - a) / max(a, b). +1 good, 0 overlapping, -1 wrong."""
    X = np.asarray(X, float); labels = np.asarray(labels)
    D = np.sqrt(np.maximum(_dist2(X, X), 0))
    sil = np.zeros(len(X))
    for i in range(len(X)):
        same = labels == labels[i]; same[i] = False
        a = D[i, same].mean() if same.any() else 0.0
        b = min((D[i, labels == c].mean()
                 for c in np.unique(labels) if c != labels[i]), default=0.0)
        sil[i] = (b - a) / (max(a, b) + 1e-12)
    return sil.mean()

import torch

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    from sklearn.datasets import make_blobs

    X, ytrue = make_blobs(n_samples=600, centers=4, cluster_std=0.8, random_state=SEED)

    km = KMeansNumPy(k=4).fit(X)
    print(f"k-means++   inertia={km.inertia_:.1f}  silhouette={silhouette_score(X, km.labels_):.3f}")

    mb = MiniBatchKMeansNumPy(k=4).fit(X)
    mb_in = _dist2(X, mb.centroids)[np.arange(len(X)), mb.labels_].sum()
    print(f"mini-batch  inertia={mb_in:.1f}")

    C, lab = kmeans_torch(X, k=4)
    t_in = ((X - C[lab]) ** 2).sum()
    print(f"torch       inertia={t_in:.1f}")

    print("\nElbow (inertia vs k):")
    for k in range(2, 7):
        print(f"  k={k}: inertia={KMeansNumPy(k=k, n_init=3).fit(X).inertia_:.1f}")


class KMeansNumPy:
    def __init__(self, k=3, init="kmeans++", n_iters=100, n_init=10, tol=1e-6, seed=SEED):
        self.k, self.init, self.n_iters = k, init, n_iters
        self.n_init, self.tol, self.seed = n_init, tol, seed
        self.centroids = None; self.labels_ = None; self.inertia_ = np.inf

    def _init_centroids(self, X, rng):
        if self.init == "random":
            return X[rng.choice(len(X), self.k, replace=False)]
        # --- k-means++: pick spread-out seeds proportional to D^2 ---
        C = [X[rng.integers(len(X))]]
        for _ in range(1, self.k):
            d2 = _dist2(X, np.array(C)).min(1)        # dist to nearest chosen
            probs = d2 / d2.sum()                     # farther -> likelier
            C.append(X[rng.choice(len(X), p=probs)])
        return np.array(C)

    def _fit_once(self, X, rng):
        C = self._init_centroids(X, rng).astype(float)
        for _ in range(self.n_iters):
            labels = _dist2(X, C).argmin(1)           # E-step: assign
            newC = np.array([X[labels == j].mean(0) if np.any(labels == j)
                             else C[j] for j in range(self.k)])   # M-step: update
            if np.linalg.norm(newC - C) < self.tol:
                C = newC; break
            C = newC
        inertia = _dist2(X, C)[np.arange(len(X)), labels].sum()
        return C, labels, inertia

    def fit(self, X):
        X = np.asarray(X, float)
        rng = np.random.default_rng(self.seed)
        for _ in range(self.n_init):                  # multiple restarts
            C, labels, inertia = self._fit_once(X, rng)
            if inertia < self.inertia_:
                self.centroids, self.labels_, self.inertia_ = C, labels, inertia
        return self

    def predict(self, X):
        return _dist2(np.asarray(X, float), self.centroids).argmin(1)


class MiniBatchKMeansNumPy:
    """Streaming/online update — each step uses a random mini-batch."""

    def __init__(self, k=3, batch=64, n_iters=300, seed=SEED):
        self.k, self.batch, self.n_iters, self.seed = k, batch, n_iters, seed

    def fit(self, X):
        X = np.asarray(X, float)
        rng = np.random.default_rng(self.seed)
        C = X[rng.choice(len(X), self.k, replace=False)].astype(float)
        counts = np.zeros(self.k)
        for _ in range(self.n_iters):
            b = X[rng.choice(len(X), self.batch, replace=False)]
            lab = _dist2(b, C).argmin(1)
            for j in range(self.k):
                pts = b[lab == j]
                for x in pts:                         # per-center running mean
                    counts[j] += 1
                    C[j] += (x - C[j]) / counts[j]
        self.centroids = C
        self.labels_ = _dist2(X, C).argmin(1)
        return self

    def predict(self, X):
        return _dist2(np.asarray(X, float), self.centroids).argmin(1)

## 5. PyTorch implementation (GPU-friendly)

In [ ]:
# ===== actual implementation from kmeans.py =====
def kmeans_torch(X, k=3, n_iters=100, seed=SEED):
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    g = torch.Generator(device="cpu").manual_seed(seed)
    Xt = torch.as_tensor(X, dtype=torch.float32, device=dev)
    idx = torch.randperm(len(Xt), generator=g)[:k]
    C = Xt[idx].clone()
    for _ in range(n_iters):
        D = torch.cdist(Xt, C)                        # (n, k)
        labels = D.argmin(1)
        newC = torch.stack([Xt[labels == j].mean(0) if (labels == j).any() else C[j]
                            for j in range(k)])
        if torch.allclose(newC, C, atol=1e-6):
            C = newC; break
        C = newC
    return C.cpu().numpy(), labels.cpu().numpy()

## 6. Train — compare variants and the elbow

In [ ]:
demo()

## 7. Visualization — clusters, centroids, and the elbow curve

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
import kmeans as M

X, _ = make_blobs(n_samples=600, centers=4, cluster_std=0.8, random_state=0)
km = M.KMeansNumPy(k=4).fit(X)
ks = range(2, 9)
inertias = [M.KMeansNumPy(k=k, n_init=3).fit(X).inertia_ for k in ks]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(X[:,0], X[:,1], c=km.labels_, s=10, cmap="tab10")
ax[0].scatter(km.centroids[:,0], km.centroids[:,1], c="k", marker="X", s=160)
ax[0].set_title("k-means++ (k=4)")
ax[1].plot(list(ks), inertias, "o-"); ax[1].axvline(4, ls="--", c="r")
ax[1].set_xlabel("k"); ax[1].set_ylabel("inertia"); ax[1].set_title("Elbow")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- Always use **k-means++** + several restarts (we take the best inertia).
- It minimizes *Euclidean* inertia → spherical bias; for elongated/varied
  clusters use GMM (soft, elliptical) or DBSCAN (density, arbitrary shapes).
- **Mini-batch** trades a little accuracy for big speed at scale.

**Next:** soft, probabilistic clusters with full covariances → Gaussian Mixture
Models (EM).